# Module 6: Visualization & exploratory data analysis (EDA)

**Goal:** *see* your data. Pick the right chart, draw it honestly, and use charts and a few numbers to explore a dataset
before modelling anything.

**You will learn:** matplotlib's figure/axes model · histograms, bars, lines, scatter, box plots · seaborn shortcuts ·
small multiples · correlation · misleading charts · an EDA checklist

**Maths in this module:** distributions and skew · covariance and the correlation coefficient *r* · why correlation ≠ causation ·
log scales

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="colorblind")   # clean look, colour-blind-friendly colours

students = pd.read_csv(DATA / "students.csv")
rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"])

## 1. Why look at data at all? Anscombe's quartet

These four small datasets have **the same** mean, variance, correlation (0.82) and best-fit line.
Summary numbers can't tell them apart. Charts can instantly.

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
quartet = {
    "I":   (x123, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (x123, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (x123, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharex=True, sharey=True)
for ax, (name, (x, y)) in zip(axes, quartet.items()):
    x, y = np.array(x), np.array(y)
    slope, intercept = np.polyfit(x, y, 1)
    ax.scatter(x, y)
    ax.plot([3, 20], [slope * 3 + intercept, slope * 20 + intercept], color="tab:red")
    ax.set_title(f"{name}: r = {np.corrcoef(x, y)[0, 1]:.2f}")
plt.tight_layout()
plt.show()

**Lesson:** always plot your data before trusting a statistic.

## 2. matplotlib's model: Figure and Axes

* A **Figure** is the whole canvas. An **Axes** is one chart on it (with its x-axis, y-axis, title, data).
* Create both with `fig, ax = plt.subplots()`, draw on `ax`, then label *everything*.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(students["exam_score"], bins=20, edgecolor="white")
ax.set_title("Exam scores are roughly bell-shaped")
ax.set_xlabel("Exam score (0–100)")
ax.set_ylabel("Number of students")
plt.show()

## 3. Choosing a chart: ask "what is my question about?"

| question about… | chart | call |
|---|---|---|
| **one number**, its shape | histogram / density | `ax.hist`, `sns.histplot` |
| **a category**, how big is each | bar chart | `ax.bar`, `df.plot.bar`, `sns.barplot` |
| **compare distributions** across groups | box plot (or violin) | `sns.boxplot` |
| **how a number changes over time** | line chart | `ax.plot`, `df.plot.line` |
| **two numbers**, are they related | scatter plot | `ax.scatter`, `sns.scatterplot` |
| **many numbers at once** | correlation heatmap | `sns.heatmap(df.corr())` |

Avoid pie charts and 3-D charts: humans compare lengths far better than angles or volumes.

### One number: histogram
The histogram divides the range into **bins** and counts values in each. Too few bins hide shape; too many show noise.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, bins in zip(axes, [5, 20, 100]):
    ax.hist(rides["rides"], bins=bins, edgecolor="white")
    ax.set_title(f"{bins} bins")
    ax.set_xlabel("Daily rides")
axes[0].set_ylabel("Days")
plt.tight_layout()
plt.show()

### A category: bar chart

In [ ]:
avg_by_major = students.groupby("major")["exam_score"].mean().sort_values()

fig, ax = plt.subplots(figsize=(6, 3.5))
avg_by_major.plot.barh(ax=ax)
ax.set_xlabel("Average exam score")
ax.set_title("Average exam score by major")
plt.show()

### Over time: line chart

Daily data is noisy; a **7-day rolling mean** reveals the trend (each point is the average of the last 7 days).

In [ ]:
daily = rides.set_index("date")["rides"]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily.index, daily, alpha=0.35, label="daily")
ax.plot(daily.index, daily.rolling(7).mean(), linewidth=2, label="7-day average")
ax.set_title("Bike-share rides, 2024")
ax.set_ylabel("Rides per day")
ax.legend()
plt.show()

### Two numbers: scatter plot (with a third as colour)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=rides, x="temp_c", y="rides", hue="is_weekend", alpha=0.7, ax=ax)
ax.set_xlabel("Mean temperature (°C)")
ax.set_ylabel("Rides per day")
ax.set_title("Warmer days, more rides; weekends are lower")
plt.show()

### Comparing groups: box plot

The box spans $Q_1$–$Q_3$ (the middle 50%), the line inside is the median, whiskers reach about 1.5 IQR, and dots beyond are outliers.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=students, x="major", y="exam_score", ax=ax)
ax.set_title("Exam score by major")
ax.set_xlabel("")
plt.show()

### Small multiples: the same chart, one per group

Instead of cramming everything into one plot, repeat a simple chart per group on shared axes. It is one of the most effective
ideas in data visualization.

In [ ]:
g = sns.FacetGrid(students, col="major", col_wrap=2, height=2.6, sharex=True, sharey=True)
g.map_dataframe(sns.scatterplot, x="study_hours_week", y="exam_score", alpha=0.7)
g.set_axis_labels("Study hours per week", "Exam score")
g.set_titles("{col_name}")
plt.show()

## 🧮 4. Maths link: distributions and skew

A histogram is a picture of a **distribution**: how often each value occurs. Common shapes:

* **symmetric / bell-shaped (normal):** mean ≈ median (exam scores)
* **right-skewed:** a long tail of large values, mean > median (incomes, rainfall, response times)
* **bimodal:** two peaks, often two hidden groups mixed together

Right-skewed data often looks much more symmetric on a **log scale** (recall $\log(ab)=\log a+\log b$: equal *ratios* get equal spacing).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
rain = rides.loc[rides["rain_mm"] > 0, "rain_mm"]
axes[0].hist(rain, bins=25, edgecolor="white"); axes[0].set_title("Rainfall on rainy days: right-skewed")
axes[1].hist(np.log10(rain), bins=25, edgecolor="white"); axes[1].set_title("…the same data on a log10 scale")
for ax in axes: ax.set_ylabel("Days")
axes[0].set_xlabel("mm"); axes[1].set_xlabel("log10(mm)")
plt.tight_layout(); plt.show()
print("mean vs median (mm):", rain.mean().round(2), rain.median().round(2))

## 🧮 5. Maths link: correlation

How strongly do two numbers move together? First the **covariance**, the average product of the two deviations from their means:

$$\operatorname{cov}(x,y)=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})$$

It's positive when $x$ and $y$ tend to be above (or below) their means *together*. Its size depends on units, so divide by both standard
deviations to get the **correlation coefficient**:

$$r=\frac{\operatorname{cov}(x,y)}{s_x\,s_y}\qquad -1\le r\le 1$$

* $r\approx 1$: strong upward line · $r\approx -1$: strong downward line · $r\approx 0$: *no linear* relationship
* $r^2$ is the share of the variation in $y$ "explained" by a straight line in $x$.

In [ ]:
x, y = students["study_hours_week"], students["exam_score"]

cov = ((x - x.mean()) * (y - y.mean())).sum() / (len(x) - 1)
r_by_hand = cov / (x.std() * y.std())
print("by hand:", round(r_by_hand, 4), "  pandas:", round(x.corr(y), 4), "  r² =", round(r_by_hand ** 2, 3))

In [ ]:
corr = students.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation matrix (students)")
plt.show()

### Limits of *r* (read before quoting any correlation)

1. **Only linear.** A perfect U-shape can have $r=0$. Always look at the scatter plot (Anscombe!).
2. **Outliers** can create or destroy a correlation.
3. **Correlation ≠ causation.** Ice-cream sales and drownings are correlated; both are driven by hot weather (a *confounder*).
4. **Groups can reverse a trend (Simpson's paradox):**

In [ ]:
rng = np.random.default_rng(0)
parts = []
for name, mx, offset in [("Group A", 2, 10), ("Group B", 6, 4)]:
    xs = rng.normal(mx, 1, 100)
    ys = offset + 1.0 * (xs - mx) + rng.normal(0, 0.6, 100)
    parts.append(pd.DataFrame({"x": xs, "y": ys, "group": name}))
sim = pd.concat(parts, ignore_index=True)

print("overall r:", round(sim["x"].corr(sim["y"]), 2))
print(sim.groupby("group").apply(lambda g: round(g["x"].corr(g["y"]), 2), include_groups=False))

fig, ax = plt.subplots(figsize=(5.5, 4))
sns.scatterplot(data=sim, x="x", y="y", hue="group", alpha=0.7, ax=ax)
ax.set_title("Overall trend down, but up within each group")
plt.show()

The overall trend is negative, yet within each group it is positive. The lesson: **look for hidden groups** before you interpret a relationship.

## 6. How charts mislead (and how not to)

The same two numbers, drawn two ways:

In [ ]:
means = students.groupby("part_time_job")["exam_score"].mean()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].bar(means.index, means.values); axes[0].set_ylim(65, 69.5)
axes[0].set_title("Truncated axis: looks like a huge gap")
axes[1].bar(means.index, means.values); axes[1].set_ylim(0, 100)
axes[1].set_title("Axis from 0: a modest difference")
for ax in axes: ax.set_ylabel("Mean exam score")
plt.tight_layout(); plt.show()

**A checklist for honest charts**

- Bars start at **zero** (lines and scatter plots don't have to)
- Title says what the reader should notice; axes have **labels and units**
- Don't use two y-axes to imply a link between unrelated series
- Same scale across panels you want compared (`sharey=True`)
- Colours that work for colour-blind readers; direct labels or a legend
- Show the data (points, distributions), not just averages
- Say what's **not** shown (sample size, missing data)

## 7. A mini-EDA recipe

Use this order for any new dataset. It builds directly into your project's EDA section.

1. **Shape & types:** `shape`, `info`, `describe`
2. **One variable at a time:** histogram for numbers, `value_counts` / bar for categories → shape, outliers, oddities
3. **Two variables:** scatter (number × number), box (number × category), grouped bars, correlation matrix
4. **Three or more:** colour / facets; look for confounders and hidden groups
5. **Write down** what you see and any new questions. These notes become the narrative of your report.

A compact worked example on the rides data:

In [ ]:
# 1-2: one variable
print(rides["rides"].describe().round(0))

# 3: relationships with the target
print(rides[["temp_c", "rain_mm", "is_weekend", "rides"]].corr().round(2)["rides"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
sns.histplot(rides["rides"], bins=25, ax=axes[0]); axes[0].set_title("Distribution of daily rides")
sns.boxplot(data=rides, x="is_weekend", y="rides", ax=axes[1]); axes[1].set_title("Weekday (0) vs weekend (1)")
sns.scatterplot(data=rides, x="rain_mm", y="rides", alpha=0.5, ax=axes[2]); axes[2].set_title("Rain vs rides")
plt.tight_layout(); plt.show()

**Notes we might write:** daily rides are roughly bell-shaped, centred near 1,500. Temperature is by far the strongest relationship
(r ≈ 0.96); weekends are modestly lower (r ≈ −0.15). Rain shows almost **no** overall correlation (r ≈ −0.02), but is that because rain
truly barely matters, or because temperature swamps it and the rain effect is small? *New questions:* compare rainy and dry days
**at similar temperatures**, and check whether the temperature effect is the same on weekends. (A first look is never the final answer.)

## 8. Saving figures

For reports and your project's `plots.py`, wrap the chart in a function that returns the Axes and a second helper that saves it:

In [ ]:
from pathlib import Path
import tempfile

def plot_rides_vs_temp(df, ax=None):
    """Scatter of daily rides against temperature."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    sns.scatterplot(data=df, x="temp_c", y="rides", alpha=0.6, ax=ax)
    ax.set_xlabel("Mean temperature (°C)")
    ax.set_ylabel("Rides per day")
    ax.set_title("Rides vs temperature")
    return ax

ax = plot_rides_vs_temp(rides)
out = Path(tempfile.mkdtemp()) / "rides_vs_temp.png"
ax.figure.savefig(out, dpi=150, bbox_inches="tight")      # dpi=150+ is crisp enough for reports
print("saved", out.name, out.stat().st_size > 0)
plt.show()

## Recap

* Plot before you trust a number (Anscombe). Pick the chart from the question.
* `fig, ax = plt.subplots()` → draw → **label everything**. seaborn gives quick, good-looking defaults.
* *r* measures *linear* association only; look for outliers, hidden groups, confounders.
* Be honest: zero baselines for bars, clear labels, no over-claiming.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** in the README.